# NB53 — 63k Genis Veri: Optimizasyon, Ensemble, Kalibrasyon

Plan: `docs/PLAN_63K_ENTEGRASYON.md` ADIM 4. NB52'nin champion recetesini (LGBM +
native_nan + scale_pos_weight + top200 feature + meta-predictor dahil + no_fe,
cv_f1=0.9904) baslangic noktasi alip:

1. **Optuna** (`TRIALS_TREE=100`, `config.py`'dan) -- CV mean F1 objective, test'e dokunulmaz.
2. **Stacking** -- A1'in en iyi 4-5 base modeli, pairwise korelasyon (<0.85 hedef, H5),
   meta = LR VE GBM (H4).
3. **Kalibrasyon** -- Platt/Isotonic/Venn-Abers, ECE once-sonra.
4. **Threshold finalizasyonu** -- F1-max vs MCC-max, secim gerekcesi.
5. **Non-missense ek-veri ablasyonu** -- n~2490 ek veri CV F1'i artiriyor mu?

Test seti **hala acilmiyor** -- NB54'e kadar. Tum kararlar 5-fold stratified CV'den.

In [1]:
# Cell 1: Imports & Config
import os, sys, json, time, warnings
import torch  # ONCE torch import et -- macOS'ta OpenMP/BLAS init sirasi SIGSEGV'ini onlemek icin (NB52 smoke test'te dogrulandi)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, TRIALS_TREE
from src import columns_63k as C63
from src.metrics import optimize_threshold, compute_all_metrics

np.random.seed(SEED)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, matthews_corrcoef, average_precision_score, brier_score_loss
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from imblearn.ensemble import BalancedBaggingClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', '63k_genis')
RESULTS_PREP_DIR = os.path.join(PROJECT_ROOT, 'results', 'v31_63k_prep')
RESULTS_BASELINE_DIR = os.path.join(PROJECT_ROOT, 'results', 'v33_63k_baseline')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v34_63k_optimization')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models', 'v31_63k')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

MISSENSE_PARQUET = os.path.join(PARQUET_DIR, 'missense_63k.parquet')
df = pd.read_parquet(MISSENSE_PARQUET)
print('missense_63k.parquet yuklendi:', df.shape)

# NB52 champion recete + secili feature seti
with open(os.path.join(RESULTS_BASELINE_DIR, 'nb52_champion_recipe.json')) as f:
    champion_recipe = json.load(f)
with open(os.path.join(RESULTS_BASELINE_DIR, 'nb52_champion_feature_list.json')) as f:
    champion_features = json.load(f)

FEATURE_COLS = champion_features['feature_cols']  # top200 (xgb-importance, NB52 A4)
CAT_COLS = champion_features['cat_cols']
print(f"Champion recete: {champion_recipe['model_family']} + {champion_recipe['missing_strategy']} + "
      f"{champion_recipe['class_balancing']} + {champion_recipe['feature_set']} ({len(FEATURE_COLS)} feature)")
print(f"NB52 referans cv_f1={champion_recipe['cv_f1_final']:.4f}")

y = df['Label'].astype(int)
groups_gene = df[C63.GENE_GROUP_COL]
prevalence = y.mean()
FLOOR_F1 = C63.floor_f1(prevalence)
spw = (y == 0).sum() / (y == 1).sum()
print(f'n={len(df)}, prevalans={prevalence:.4f}, floor-F1={FLOOR_F1:.4f}, scale_pos_weight={spw:.4f}')
print(f'TRIALS_TREE={TRIALS_TREE} (config.py, degistirilmedi)')

RESULTS_LOG = []

missense_63k.parquet yuklendi: (60970, 533)
Champion recete: lgbm + native_nan + scale_pos_weight + top200 (200 feature)
NB52 referans cv_f1=0.9897
n=60970, prevalans=0.3727, floor-F1=0.5430, scale_pos_weight=1.6833
TRIALS_TREE=100 (config.py, degistirilmedi)


## Ortak Degerlendirme Altyapisi (NB52'den yeniden kullanilan desen)

`impute_train_only` / `encode_categoricals_train_only` NB52 ile birebir ayni --
train-only fit, split sonrasi transform (CLAUDE.md sozlesme #1). `evaluate_recipe`
NB52'nin fonksiyonuna esdeger ama burada tekli model degerlendirme + OOF tahmin
uretimini (stacking icin) birlikte yapan `cv_predict_oof` eklendi.

In [2]:
# Cell 2: Ortak yardimci fonksiyonlar (NB52 ile ayni sozlesme)
def encode_categoricals_train_only(X_tr, X_val, cat_cols):
    if not cat_cols:
        return X_tr, X_val
    X_tr = X_tr.copy(); X_val = X_val.copy()
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_tr[cat_cols] = X_tr[cat_cols].astype(str).fillna('__NA__')
    X_val[cat_cols] = X_val[cat_cols].astype(str).fillna('__NA__')
    X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols])
    X_val[cat_cols] = enc.transform(X_val[cat_cols])
    return X_tr, X_val


def stringify_categoricals(X_tr, X_val, cat_cols):
    if not cat_cols:
        return X_tr, X_val
    X_tr = X_tr.copy(); X_val = X_val.copy()
    X_tr[cat_cols] = X_tr[cat_cols].astype(str).fillna('__NA__')
    X_val[cat_cols] = X_val[cat_cols].astype(str).fillna('__NA__')
    return X_tr, X_val


def impute_train_only(X_tr, X_val, numeric_subset):
    if not numeric_subset:
        return X_tr, X_val
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X_tr = X_tr.copy(); X_val = X_val.copy()
    X_tr[numeric_subset] = imputer.fit_transform(X_tr[numeric_subset])
    X_val[numeric_subset] = imputer.transform(X_val[numeric_subset])
    return X_tr, X_val


def prep_fold(frame, cols, cat_cols, family, train_idx, val_idx, needs_impute=True):
    """Tek bir CV fold icin train/val hazirlar (impute/encode, model ailesine gore)."""
    X_tr = frame.iloc[train_idx][cols].copy()
    X_val = frame.iloc[val_idx][cols].copy()
    numeric_subset = [c for c in cols if c not in cat_cols]

    if needs_impute and family != 'catboost':
        X_tr, X_val = impute_train_only(X_tr, X_val, numeric_subset)
    if family in ('xgboost', 'random_forest', 'balanced_bagging', 'logistic'):
        X_tr, X_val = encode_categoricals_train_only(X_tr, X_val, cat_cols)
        eff_cat_cols = []
    elif family == 'catboost':
        X_tr, X_val = stringify_categoricals(X_tr, X_val, cat_cols)
        eff_cat_cols = cat_cols
    else:
        eff_cat_cols = cat_cols
    return X_tr, X_val, eff_cat_cols


def build_model(family, cat_cols=None, params=None, scale_pos_weight=None):
    params = params or {}
    if family == 'lgbm':
        p = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, verbosity=-1, scale_pos_weight=scale_pos_weight)
        p.update(params)
        return lgb.LGBMClassifier(**p)
    if family == 'xgboost':
        p = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, eval_metric='logloss', verbosity=0, scale_pos_weight=scale_pos_weight)
        p.update(params)
        return xgb.XGBClassifier(**p)
    if family == 'catboost':
        p = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=False, scale_pos_weight=scale_pos_weight)
        if cat_cols:
            p['cat_features'] = cat_cols
        p.update(params)
        return CatBoostClassifier(**p)
    if family == 'random_forest':
        p = dict(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight='balanced')
        p.update(params)
        return RandomForestClassifier(**p)
    if family == 'balanced_bagging':
        p = dict(n_estimators=10, random_state=SEED, n_jobs=1)  # n_jobs=1: NB52 smoke test'te deadlock bulundu
        p.update(params)
        return BalancedBaggingClassifier(
            estimator=lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, random_state=SEED, verbosity=-1, n_jobs=1), **p)
    raise ValueError(family)


def cv_score(frame, cols, cat_cols, family, y_arr, params=None, n_splits=5, needs_impute=True, scale_pos_weight=None):
    """Basit CV skoru (Optuna objective icin) -- sadece StratifiedKFold, ortalama F1 doner."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    f1s = []
    for train_idx, val_idx in skf.split(frame, y_arr):
        X_tr, X_val, eff_cat = prep_fold(frame, cols, cat_cols, family, train_idx, val_idx, needs_impute)
        y_tr, y_val = y_arr.iloc[train_idx], y_arr.iloc[val_idx]
        model = build_model(family, cat_cols=eff_cat, params=params, scale_pos_weight=scale_pos_weight)
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        f1s.append(f1_score(y_val, (proba >= 0.5).astype(int)))
    return float(np.mean(f1s))


def cv_predict_oof(frame, cols, cat_cols, family, y_arr, params=None, n_splits=5, needs_impute=True,
                    scale_pos_weight=None, return_group_f1=False):
    """5-fold OOF tahmin uretir (stacking base-learner girdisi) + genegroup F1 (istege bagli)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_proba = np.zeros(len(frame))
    for train_idx, val_idx in skf.split(frame, y_arr):
        X_tr, X_val, eff_cat = prep_fold(frame, cols, cat_cols, family, train_idx, val_idx, needs_impute)
        y_tr = y_arr.iloc[train_idx]
        model = build_model(family, cat_cols=eff_cat, params=params, scale_pos_weight=scale_pos_weight)
        model.fit(X_tr, y_tr)
        oof_proba[val_idx] = model.predict_proba(X_val)[:, 1]

    result = {'oof_proba': oof_proba, 'cv_f1': f1_score(y_arr, (oof_proba >= 0.5).astype(int))}
    if return_group_f1:
        gkf = GroupKFold(n_splits=n_splits)
        oof_group = np.zeros(len(frame))
        for train_idx, val_idx in gkf.split(frame, y_arr, groups=groups_gene.iloc[frame.index]):
            X_tr, X_val, eff_cat = prep_fold(frame, cols, cat_cols, family, train_idx, val_idx, needs_impute)
            y_tr = y_arr.iloc[train_idx]
            model = build_model(family, cat_cols=eff_cat, params=params, scale_pos_weight=scale_pos_weight)
            model.fit(X_tr, y_tr)
            oof_group[val_idx] = model.predict_proba(X_val)[:, 1]
        result['genegroup_f1'] = f1_score(y_arr, (oof_group >= 0.5).astype(int))
    return result


def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE: bin'lere ayirip tahmin-gerceklesme farkinin agirlikli ortalamasi."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1] if i < n_bins - 1 else y_prob <= bins[i + 1])
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(conf - acc)
    return ece

## 1. Optuna — Champion Recete Hiperparametre Optimizasyonu

`TRIALS_TREE=100` (`config.py`'dan, degistirilmedi). Objective: 5-fold CV mean F1
(champion feature seti + native_nan + scale_pos_weight sabit, sadece LGBM
hiperparametreleri aranir). Test setine dokunulmaz.

In [3]:
# Cell 3: Optuna -- LGBM hiperparametre optimizasyonu
FAMILY = champion_recipe['model_family']  # 'lgbm'
assert FAMILY == 'lgbm', 'Bu Optuna objective su an sadece lgbm icin parametrelenmis'

def objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    return cv_score(df, FEATURE_COLS, CAT_COLS, FAMILY, y, params=params,
                     needs_impute=False, scale_pos_weight=spw)

t0 = time.time()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=TRIALS_TREE, show_progress_bar=False)
optuna_elapsed = time.time() - t0

print(f'Optuna tamamlandi: {TRIALS_TREE} trial, {optuna_elapsed:.1f}s')
print(f'En iyi CV F1: {study.best_value:.4f} (NB52 champion: {champion_recipe["cv_f1_final"]:.4f})')
print(f'En iyi parametreler: {study.best_params}')

BEST_LGBM_PARAMS = study.best_params
optuna_delta = study.best_value - champion_recipe['cv_f1_final']
print(f'Optuna kazanci: {optuna_delta:+.4f}')

with open(os.path.join(RESULTS_DIR, 'nb53_optuna_best_params.json'), 'w') as f:
    json.dump({'best_params': BEST_LGBM_PARAMS, 'best_cv_f1': study.best_value,
               'n_trials': TRIALS_TREE, 'elapsed_seconds': round(optuna_elapsed, 1)}, f, indent=2)

RESULTS_LOG.append({'name': 'B1_optuna_lgbm', 'cv_f1': study.best_value, 'note': f'{TRIALS_TREE} trials'})

Optuna tamamlandi: 100 trial, 3336.7s
En iyi CV F1: 0.9907 (NB52 champion: 0.9897)
En iyi parametreler: {'num_leaves': 150, 'max_depth': 4, 'learning_rate': 0.11455183364566728, 'n_estimators': 488, 'min_child_samples': 75, 'subsample': 0.8566043114169304, 'colsample_bytree': 0.6341508096539421, 'reg_alpha': 7.450048837777351e-08, 'reg_lambda': 0.012235743786081084}
Optuna kazanci: +0.0010


## 2. Stacking — Base Modeller + Pairwise Korelasyon (H5) + Meta LR vs GBM (H4)

NB52 A1'in 4-5 en iyi ailesi (Optuna'lanmis LGBM dahil) OOF tahminleriyle bir meta-
matris olusturur. **H5**: base'ler arasi pairwise korelasyon <0.85 mi (heterojenlik)?
**H4**: meta-learner LR mi GBM mi kazanir (63k'da meta-matris n≈61k, eski overfit
gerekcesi düşmüş olabilir mi)?

In [4]:
# Cell 4: Stacking base modelleri -- OOF tahmin matrisi
BASE_FAMILIES = {
    'lgbm_optuna': ('lgbm', BEST_LGBM_PARAMS, False),
    'xgboost': ('xgboost', {}, False),
    'catboost': ('catboost', {}, True),   # needs_impute=True (catboost native nan ama impute etsek de zarari yok, tutarlilik icin varsayilanla birak)
    'random_forest': ('random_forest', {}, True),
    'balanced_bagging': ('balanced_bagging', {}, True),
}

oof_matrix = {}
base_cv_f1 = {}
base_gene_f1 = {}
for name, (family, params, needs_impute) in BASE_FAMILIES.items():
    t0 = time.time()
    res = cv_predict_oof(df, FEATURE_COLS, CAT_COLS, family, y, params=params,
                          needs_impute=needs_impute, scale_pos_weight=spw, return_group_f1=True)
    oof_matrix[name] = res['oof_proba']
    base_cv_f1[name] = res['cv_f1']
    base_gene_f1[name] = res['genegroup_f1']
    print(f'[{name}] oof_f1={res["cv_f1"]:.4f} gene_f1={res["genegroup_f1"]:.4f} ({time.time()-t0:.1f}s)')

oof_df = pd.DataFrame(oof_matrix)
corr_matrix = oof_df.corr()
print()
print('=== Pairwise korelasyon matrisi (OOF proba) ===')
print(corr_matrix.round(3).to_string())

max_pairwise_corr = corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool)).max().max()
print()
print(f'Maksimum pairwise korelasyon: {max_pairwise_corr:.4f}')
h5_verdict = 'DOGRULANDI (heterojen, <0.85)' if max_pairwise_corr < 0.85 else 'CURUTULDU (homojen, >=0.85)'
print(f'H5 (heterojen base korelasyon<0.85) -> {h5_verdict}')

corr_matrix.to_csv(os.path.join(RESULTS_DIR, 'nb53_base_correlation.csv'))
oof_df.to_parquet(os.path.join(RESULTS_DIR, 'nb53_oof_matrix.parquet'))

[lgbm_optuna] oof_f1=0.9907 gene_f1=0.9886 (57.0s)
[xgboost] oof_f1=0.9895 gene_f1=0.9888 (159.4s)
[catboost] oof_f1=0.9892 gene_f1=0.9875 (420.3s)
[random_forest] oof_f1=0.9871 gene_f1=0.9864 (251.8s)
[balanced_bagging] oof_f1=0.9883 gene_f1=0.9875 (363.7s)

=== Pairwise korelasyon matrisi (OOF proba) ===
                  lgbm_optuna  xgboost  catboost  random_forest  balanced_bagging
lgbm_optuna             1.000    0.998     0.997          0.991             0.997
xgboost                 0.998    1.000     0.998          0.993             0.999
catboost                0.997    0.998     1.000          0.995             0.999
random_forest           0.991    0.993     0.995          1.000             0.994
balanced_bagging        0.997    0.999     0.999          0.994             1.000

Maksimum pairwise korelasyon: 0.9988
H5 (heterojen base korelasyon<0.85) -> CURUTULDU (homojen, >=0.85)


In [5]:
# Cell 5: Meta-learner -- LR vs GBM (H4), 5-fold nested CV (OOF uzerinde meta-CV)
def evaluate_meta(meta_model_fn, oof_df, y_arr, n_splits=5):
    """OOF matrisi uzerinde meta-learner'i degerlendirir (nested: meta de kendi CV'siyle)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    meta_oof = np.zeros(len(oof_df))
    for train_idx, val_idx in skf.split(oof_df, y_arr):
        X_tr, X_val = oof_df.iloc[train_idx], oof_df.iloc[val_idx]
        y_tr = y_arr.iloc[train_idx]
        model = meta_model_fn()
        model.fit(X_tr, y_tr)
        meta_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    return meta_oof, f1_score(y_arr, (meta_oof >= 0.5).astype(int))

meta_lr_fn = lambda: LogisticRegression(penalty='l2', class_weight='balanced', max_iter=2000, random_state=SEED)
meta_gbm_fn = lambda: lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=SEED, verbosity=-1)

meta_lr_oof, meta_lr_f1 = evaluate_meta(meta_lr_fn, oof_df, y)
meta_gbm_oof, meta_gbm_f1 = evaluate_meta(meta_gbm_fn, oof_df, y)

print(f'Meta=LR   cv_f1={meta_lr_f1:.4f}')
print(f'Meta=GBM  cv_f1={meta_gbm_f1:.4f}')
print(f'Single best base ({max(base_cv_f1, key=base_cv_f1.get)}) cv_f1={max(base_cv_f1.values()):.4f}')

h4_verdict = 'GBM KAZANDI (eski gerekce dustu)' if meta_gbm_f1 > meta_lr_f1 + 0.002 else 'LR YETERLI/USTUN (eski bulgu tutarli)'
print(f'H4 (meta=LR vs GBM) -> {h4_verdict}')

BEST_META = 'gbm' if meta_gbm_f1 > meta_lr_f1 else 'lr'
BEST_META_OOF = meta_gbm_oof if BEST_META == 'gbm' else meta_lr_oof
BEST_META_F1 = max(meta_gbm_f1, meta_lr_f1)
best_single_base_f1 = max(base_cv_f1.values())
stacking_delta = BEST_META_F1 - best_single_base_f1
print(f'Stacking kazanci (best_meta - best_single_base) = {stacking_delta:+.4f}')

RESULTS_LOG.append({'name': 'B2_stacking_lr', 'cv_f1': meta_lr_f1, 'note': 'meta=LR'})
RESULTS_LOG.append({'name': 'B2_stacking_gbm', 'cv_f1': meta_gbm_f1, 'note': 'meta=GBM'})

Meta=LR   cv_f1=0.9900
Meta=GBM  cv_f1=0.9906
Single best base (lgbm_optuna) cv_f1=0.9907
H4 (meta=LR vs GBM) -> LR YETERLI/USTUN (eski bulgu tutarli)
Stacking kazanci (best_meta - best_single_base) = -0.0001


## 3. Kalibrasyon — Platt / Isotonic + ECE

Secili kazanan (Optuna-LGBM tekli veya stacking, hangisi daha iyiyse) uzerinde
olasilik kalibrasyonu. ECE once-sonra raporlanir. Prior-shift YOK (63k'da hedef
prior yok, yarisma verisinden farkli olarak test dagilimi bilinmiyor/degismiyor).
Venn-Abers plandaydi ama ek bagimlilik (`venn-abers` paketi kurulu degil) ve bu
projede Platt/Isotonic'in zaten yeterli oldugu (bkz. NB17 S6) goz onune alinarak
kapsam disi birakildi.

In [6]:
# Cell 6: Kalibrasyon (Platt/Isotonic) + ECE once-sonra
# Onceki adimlarin en iyisini secelim (tekli Optuna-LGBM vs stacking)
CHAMPION_OOF = oof_matrix['lgbm_optuna'] if best_single_base_f1 >= BEST_META_F1 else BEST_META_OOF
CHAMPION_CV_F1 = max(best_single_base_f1, BEST_META_F1)
CHAMPION_IS_STACKING = BEST_META_F1 > best_single_base_f1
print(f'Kalibrasyon icin secilen model: {"stacking(" + BEST_META + ")" if CHAMPION_IS_STACKING else "lgbm_optuna"} (cv_f1={CHAMPION_CV_F1:.4f})')

y_arr = y.values
ece_before = expected_calibration_error(y_arr, CHAMPION_OOF)

# Platt scaling (sigmoid) -- CV icinde (nested, train-only fit)
def calibrate_platt_cv(raw_oof, y_arr, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    calibrated = np.zeros(len(raw_oof))
    for train_idx, val_idx in skf.split(raw_oof.reshape(-1, 1), y_arr):
        platt = LogisticRegression()
        platt.fit(raw_oof[train_idx].reshape(-1, 1), y_arr[train_idx])
        calibrated[val_idx] = platt.predict_proba(raw_oof[val_idx].reshape(-1, 1))[:, 1]
    return calibrated

def calibrate_isotonic_cv(raw_oof, y_arr, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    calibrated = np.zeros(len(raw_oof))
    for train_idx, val_idx in skf.split(raw_oof.reshape(-1, 1), y_arr):
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(raw_oof[train_idx], y_arr[train_idx])
        calibrated[val_idx] = iso.predict(raw_oof[val_idx])
    return calibrated

platt_oof = calibrate_platt_cv(CHAMPION_OOF, y_arr)
isotonic_oof = calibrate_isotonic_cv(CHAMPION_OOF, y_arr)

ece_platt = expected_calibration_error(y_arr, platt_oof)
ece_isotonic = expected_calibration_error(y_arr, isotonic_oof)
brier_before = brier_score_loss(y_arr, CHAMPION_OOF)
brier_platt = brier_score_loss(y_arr, platt_oof)
brier_isotonic = brier_score_loss(y_arr, isotonic_oof)

print(f'ECE  ham={ece_before:.4f}  platt={ece_platt:.4f}  isotonic={ece_isotonic:.4f}')
print(f'Brier ham={brier_before:.4f}  platt={brier_platt:.4f}  isotonic={brier_isotonic:.4f}')

calib_candidates = {'none': (CHAMPION_OOF, ece_before), 'platt': (platt_oof, ece_platt), 'isotonic': (isotonic_oof, ece_isotonic)}
BEST_CALIB = min(calib_candidates, key=lambda k: calib_candidates[k][1])
CALIBRATED_OOF = calib_candidates[BEST_CALIB][0]
print(f'En iyi kalibrasyon (ECE-min): {BEST_CALIB}')

RESULTS_LOG.append({'name': 'B3_calibration', 'cv_f1': f1_score(y_arr, (CALIBRATED_OOF>=0.5).astype(int)),
                     'note': f'{BEST_CALIB}, ece={calib_candidates[BEST_CALIB][1]:.4f}'})

Kalibrasyon icin secilen model: lgbm_optuna (cv_f1=0.9907)
ECE  ham=0.0045  platt=0.0007  isotonic=0.0005
Brier ham=0.0059  platt=0.0064  isotonic=0.0056
En iyi kalibrasyon (ECE-min): isotonic


## 4. Threshold Finalizasyonu — F1-max vs MCC-max

Kalibre edilmis OOF olasiliklar uzerinde her iki threshold stratejisi de hesaplanir,
ikisi de raporlanir, secim gerekcesi yazilir.

In [7]:
# Cell 7: Threshold finalizasyonu
best_thr_f1, best_f1_at_thr = optimize_threshold(y_arr, CALIBRATED_OOF)

def optimize_threshold_mcc(y_true, y_prob):
    best_thr, best_mcc = 0.5, -1
    for thr in np.arange(0.10, 0.90, 0.01):
        mcc_val = matthews_corrcoef(y_true, (y_prob >= thr).astype(int))
        if mcc_val > best_mcc:
            best_mcc, best_thr = mcc_val, thr
    return best_thr, best_mcc

best_thr_mcc, best_mcc_at_thr = optimize_threshold_mcc(y_arr, CALIBRATED_OOF)

f1_at_mcc_thr = f1_score(y_arr, (CALIBRATED_OOF >= best_thr_mcc).astype(int))
mcc_at_f1_thr = matthews_corrcoef(y_arr, (CALIBRATED_OOF >= best_thr_f1).astype(int))

print(f'F1-max threshold={best_thr_f1:.2f}  -> F1={best_f1_at_thr:.4f}, MCC={mcc_at_f1_thr:.4f}')
print(f'MCC-max threshold={best_thr_mcc:.2f} -> MCC={best_mcc_at_thr:.4f}, F1={f1_at_mcc_thr:.4f}')

# Karar: 63k'da test dagilimi train ile ayni prevalansta bekleniyor (yarisma verisinin
# aksine, final test icin bilinen bir "ters dagilim" beyani yok) -- bu yuzden F1-max
# tercih edilir, MCC yalnizca capraz-kontrol amacli raporlanir.
FINAL_THRESHOLD = best_thr_f1
threshold_reason = ('F1-max secildi: 63k icin (yarisma verisinin aksine) bilinen bir ters-dagilim '
                     'beyani yok, train ve degerlendirme ayni prevalansta kaliyor; MCC-max capraz-kontrol.')
print()
print(f'SECILEN THRESHOLD: {FINAL_THRESHOLD:.2f}')
print(f'Gerekce: {threshold_reason}')

RESULTS_LOG.append({'name': 'B4_threshold_f1max', 'cv_f1': best_f1_at_thr, 'note': f'thr={best_thr_f1:.2f}'})
RESULTS_LOG.append({'name': 'B4_threshold_mccmax', 'cv_f1': f1_at_mcc_thr, 'note': f'thr={best_thr_mcc:.2f}, mcc={best_mcc_at_thr:.4f}'})

F1-max threshold=0.46  -> F1=0.9907, MCC=0.9852
MCC-max threshold=0.46 -> MCC=0.9852, F1=0.9907

SECILEN THRESHOLD: 0.46
Gerekce: F1-max secildi: 63k icin (yarisma verisinin aksine) bilinen bir ters-dagilim beyani yok, train ve degerlendirme ayni prevalansta kaliyor; MCC-max capraz-kontrol.


## 5. Non-Missense Ek-Veri Ablasyonu

`nonmis_63k.parquet` (n~2490, missense-disi varyantlar -- frameshift/stopgain/vs.)
egitime eklemek CV F1'i degistiriyor mu? Ortak feature kesisimiyle (champion feature
seti nonmissense'te de mevcutsa) test edilir.

In [8]:
# Cell 8: Non-missense ek-veri ablasyonu
NONMIS_PARQUET = os.path.join(PARQUET_DIR, 'nonmis_63k.parquet')
df_nonmis = pd.read_parquet(NONMIS_PARQUET)
print(f'nonmis_63k.parquet yuklendi: {df_nonmis.shape}')

common_features = [c for c in FEATURE_COLS if c in df_nonmis.columns]
missing_in_nonmis = [c for c in FEATURE_COLS if c not in df_nonmis.columns]
print(f'Ortak feature: {len(common_features)}/{len(FEATURE_COLS)} (nonmissense\'de eksik: {len(missing_in_nonmis)})')

if len(common_features) < len(FEATURE_COLS) * 0.8:
    print('UYARI: nonmissense verisinde feature kesisimi cok dusuk, ablasyon guvenilmez olabilir.')

df_combined = pd.concat([df[['Label', C63.GENE_GROUP_COL] + common_features],
                          df_nonmis[['Label', C63.GENE_GROUP_COL] + common_features]], ignore_index=True)
y_combined = df_combined['Label'].astype(int)
cat_cols_common = [c for c in CAT_COLS if c in common_features]

print(f'Birlesik veri: n={len(df_combined)} (missense={len(df)}, +nonmissense={len(df_nonmis)})')

df_missense_frame = df[['Label', C63.GENE_GROUP_COL] + common_features]
res_missense_only = cv_predict_oof(df_missense_frame, common_features, cat_cols_common, FAMILY, y, params=BEST_LGBM_PARAMS,
                                     needs_impute=False, scale_pos_weight=spw)
res_combined = cv_predict_oof(df_combined, common_features, cat_cols_common, FAMILY, y_combined, params=BEST_LGBM_PARAMS,
                                needs_impute=False, scale_pos_weight=(y_combined==0).sum()/(y_combined==1).sum())

print(f'Sadece missense (ortak feature)  cv_f1={res_missense_only["cv_f1"]:.4f}')
print(f'+ non-missense birlesik          cv_f1={res_combined["cv_f1"]:.4f}')

nonmis_delta = res_combined['cv_f1'] - res_missense_only['cv_f1']
print(f'Fark (birlesik - sadece missense) = {nonmis_delta:+.4f}')
nonmis_verdict = 'FAYDALI, EKLE' if nonmis_delta > 0.003 else ('ZARARLI' if nonmis_delta < -0.003 else 'NOTR')
print(f'Non-missense ek-veri karari -> {nonmis_verdict}')

RESULTS_LOG.append({'name': 'B5_missense_only', 'cv_f1': res_missense_only['cv_f1'], 'note': f'{len(common_features)} ortak feature'})
RESULTS_LOG.append({'name': 'B5_combined_nonmis', 'cv_f1': res_combined['cv_f1'], 'note': f'n={len(df_combined)}'})

nonmis_63k.parquet yuklendi: (2482, 781)
Ortak feature: 200/200 (nonmissense'de eksik: 0)
Birlesik veri: n=63452 (missense=60970, +nonmissense=2482)
Sadece missense (ortak feature)  cv_f1=0.9907
+ non-missense birlesik          cv_f1=0.9890
Fark (birlesik - sadece missense) = -0.0017
Non-missense ek-veri karari -> NOTR


## Ozet + Champion Recete v2 + PDF Rapor

In [9]:
# Cell 9: Ozet tablo + champion recete v2 + PDF rapor
results_df = pd.DataFrame(RESULTS_LOG)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(RESULTS_DIR, 'nb53_optimization_results.csv'), index=False)

champion_v2 = {
    'model_family': FAMILY,
    'optuna_params': BEST_LGBM_PARAMS,
    'optuna_cv_f1': study.best_value,
    'stacking_used': CHAMPION_IS_STACKING,
    'stacking_meta': BEST_META if CHAMPION_IS_STACKING else None,
    'max_pairwise_base_corr': float(max_pairwise_corr),
    'calibration_method': BEST_CALIB,
    'ece_before': float(ece_before),
    'ece_after': float(calib_candidates[BEST_CALIB][1]),
    'final_threshold': float(FINAL_THRESHOLD),
    'threshold_strategy': 'f1max',
    'threshold_reason': threshold_reason,
    'nonmissense_included': nonmis_delta > 0.003,
    'nonmissense_delta': float(nonmis_delta),
    'final_cv_f1': float(CHAMPION_CV_F1),
    'floor_f1': float(FLOOR_F1),
}

hypothesis_scorecard_v2 = {
    'H4_meta_lr_not_gbm': h4_verdict,
    'H5_heterogeneous_base_corr_lt_085': h5_verdict,
}

print()
print('=== NB53 CHAMPION RECETE v2 ===')
for k, v in champion_v2.items():
    print(f'{k}: {v}')
print()
print('=== H4/H5 KARNESI ===')
for k, v in hypothesis_scorecard_v2.items():
    print(f'{k}: {v}')

with open(os.path.join(RESULTS_DIR, 'nb53_champion_recipe_v2.json'), 'w') as f:
    json.dump(champion_v2, f, indent=2, ensure_ascii=False)
with open(os.path.join(RESULTS_DIR, 'nb53_hypothesis_scorecard.json'), 'w') as f:
    json.dump(hypothesis_scorecard_v2, f, indent=2, ensure_ascii=False)

# Ilerleme grafigi
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df['name'], results_df['cv_f1'], marker='o', linewidth=2)
ax.axhline(FLOOR_F1, color='red', linestyle='--', label=f'Floor-F1={FLOOR_F1:.3f}')
ax.axhline(champion_recipe['cv_f1_final'], color='gray', linestyle=':', label=f'NB52 champion={champion_recipe["cv_f1_final"]:.4f}')
ax.set_ylabel('CV F1')
ax.set_title('NB53 Optimizasyon Ilerlemesi')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'nb53_optimization_progress.png'), dpi=120)
plt.close(fig)
print('Grafik kaydedildi: nb53_optimization_progress.png')

               name    cv_f1                 note
     B1_optuna_lgbm 0.990700           100 trials
     B2_stacking_lr 0.989959              meta=LR
    B2_stacking_gbm 0.990598             meta=GBM
     B3_calibration 0.990701 isotonic, ece=0.0005
 B4_threshold_f1max 0.990725             thr=0.46
B4_threshold_mccmax 0.990725 thr=0.46, mcc=0.9852
   B5_missense_only 0.990700    200 ortak feature
 B5_combined_nonmis 0.989008              n=63452

=== NB53 CHAMPION RECETE v2 ===
model_family: lgbm
optuna_params: {'num_leaves': 150, 'max_depth': 4, 'learning_rate': 0.11455183364566728, 'n_estimators': 488, 'min_child_samples': 75, 'subsample': 0.8566043114169304, 'colsample_bytree': 0.6341508096539421, 'reg_alpha': 7.450048837777351e-08, 'reg_lambda': 0.012235743786081084}
optuna_cv_f1: 0.9907003864181835
stacking_used: False
stacking_meta: None
max_pairwise_base_corr: 0.9987925210233531
calibration_method: isotonic
ece_before: 0.004512223747696455
ece_after: 0.0005040174122324336
final_

In [10]:
# Cell 10: PDF rapor
from fpdf import FPDF

class NB53Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB53 - Optimizasyon, Ensemble, Kalibrasyon Raporu', ln=True, align='C')
        self.ln(2)

    def section(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 8, title, ln=True)
        self.set_font('Helvetica', '', 9)

    def kv_table(self, d):
        for k, v in d.items():
            self.cell(0, 6, f'{k}: {v}', ln=True)
        self.ln(2)

report = NB53Report()
report.add_page()

report.section('1. NB52 Baslangic Noktasi')
report.kv_table(champion_recipe)

report.section('2. H4/H5 Karnesi')
report.kv_table(hypothesis_scorecard_v2)

report.section('3. NB53 Champion Recete v2')
report.kv_table({k: v for k, v in champion_v2.items() if k != 'optuna_params'})

report.section('4. Tum Optimizasyon Sonuclari')
for row in RESULTS_LOG:
    report.cell(0, 5, f"{row['name']}: cv_f1={row['cv_f1']:.4f} ({row.get('note', '')})", ln=True)

REPORT_PATH = os.path.join(REPORTS_DIR, 'nb53_optimization_report.pdf')
report.output(REPORT_PATH)
print(f'PDF rapor yazildi: {REPORT_PATH}')

PDF rapor yazildi: /Users/tefe/teknofest_model/teknofest_model/reports/nb53_optimization_report.pdf


## Sonraki Adim

**ADIM 5 — NB54 (`notebooks/54_63k_final.ipynb`):** Test seti **ilk ve son kez**
burada acilir. Final metrikler (hold-out + floor + confusion matrix), CV-test
farki raporu (protokol durustlugu), gen-holdout final skoru, model/artifact
kaydi (`models/v31_63k/`).